# Lista 8

## PyTorch

(6pkt + 2pkt)

Na liście znajduje się 1 zadanie. Po rozwiązaniu go, pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** — tylko wtedy przyznajemy punkty. Dodatkowo prześlij zadanie na platformie skos.

Dodatkowe zadanie oznaczone jest ⭐️ i jest warte 2pkt.

## Jeśli ćwiczenia będą się przedłuały...

Odpowiedz na ponisze pytania pisemnie i prześlij zadanie na skos. Do zobaczenia na kolejnych zajęciach! 😀

1. (W tym pytaniu nie ma złej odpowiedzi, jeśli umiesz ją uzasadnić) Zbiorem danych są obrazki, na których są napisane cyfry. Jakie transformacje moznaby zastosować na obrazkach, za pomocą których będziemy uczyli potem sieć?
2. Czy w sieci neuronowej dla ponizszego zbioru danych i dla tej konkretnej architektury potrzebne jest `Flatten`? Dlaczego?
3. W zeszłym tygodniu omawialiśmy Cross Validation. Jak wyglądałoby ono w przypadku uczenia sieci neuronowej?

## Wytrenuj sieć neuronową

Twoim celem jest przygotować kompletny pipeline do trenowania prostej sieci neuronowej w PyTorchu: od datasetu, przez dataloadery i model, aż po trening, walidację i test.

---

### Dataset i transformacje

Przygotuj dane wejściowe korzystając z `torchvision.datasets.MNIST`.

#### Kroki:

1. Zaimportuj konieczne biblioteki:

   * `torch`
   * `torchvision`
   * `torch.utils.data`
   * oraz moduł transformacji: `torchvision.transforms as transforms`

2. Pobierz dataset **MNIST** dla:

   * zbioru treningowego,
   * zbioru testowego.

3. Zastosuj transformacje:

   * **obowiązkowo:** `transforms.ToTensor()`
   * **opcjonalnie:** normalizacja
     `transforms.Normalize((0.5,), (0.5,))`

4. Podziel zbiór treningowy na dwie części przy użyciu `torch.utils.data.random_split`:

   * **90%** — train
   * **10%** — validation

5. **Nie twórz własnego datasetu.** Użyj wbudowanego `torchvision.datasets.MNIST`.

**Uwaga:** Upewnij się, że rozmiar batcha to `(batch, 1, 28, 28)` — transformacje nie mogą zmieniać tego formatu.

---

### Dataloadery

Utwórz trzy `DataLoader`-y:

* **train** — `shuffle=True`
* **validation** — `shuffle=False`
* **test** — `shuffle=False`

Z parametrami:

* `batch_size = 64`
* `num_workers = 0`
  (żeby uniknąć problemów np. na Windowsowych laptopach)

---

### Sieć neuronowa (MLP)

Zaimplementuj model dziedzicząc po `nn.Module`.

#### Architektura — dokładnie w tej kolejności:

1. `Flatten()`
2. `Linear(28*28, 256)`
3. `ReLU()`
4. `Linear(256, 128)`
5. `ReLU()`
6. `Linear(128, 10)`

**Wskazówki:**

* Warstwy umieść w `nn.Sequential`.
* Domyślna aktywacja to ReLU (ale możesz eksperymentować).

#### Dodatkowe elementy:

1. Funkcja straty: **CrossEntropyLoss**.
2. Optymalizator: **Adam**, `lr=0.001`.

---

### Pętla treningowa

Napisz pętlę uczącą, która dla każdej epoki:

#### W trybie `model.train()`:

Dla każdego batcha:

* oblicza logitsy (predykcje),
* oblicza stratę,
* wykonuje:

  * `optimizer.zero_grad()`
  * `loss.backward()`
  * `optimizer.step()`
* liczy poprawne odpowiedzi.

#### Po epce — wyświetl:

* średni **train loss**
* **train accuracy**

**Wskazówka (accuracy):**

```python
pred = logits.argmax(dim=1)
correct += (pred == y).sum().item()
```

---

### Walidacja + zapis najlepszego modelu

Dodaj funkcję walidującą:

* `model.eval()`
* `torch.no_grad()`

Policz:

* średni **val loss**
* **val accuracy**

#### Po każdej epoce:

* sprawdź, czy `val_loss` jest najlepszy w historii,
* jeśli tak — zapisz model:

```python
torch.save(model.state_dict(), "best_model.pth")
```

* wypisz komunikat:

> **"Zapisano nowy najlepszy model!"**

---

### Testowanie najlepszego modelu

1. Wczytaj najlepsze wagi:

   ```python
   model.load_state_dict(torch.load("best_model.pth"))
   ```

2. Przeprowadź ewaluację na zbiorze **testowym** (procedura taka jak przy walidacji).

3. Wyświetl **finalną test accuracy**.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Dataset + Transformacje

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_full = datasets.MNIST(root=".", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(
    root=".",
    train=False,
    download=True,
    transform=transform
)

# Podział train/val
train_size = int(0.9 * len(train_full))
val_size = len(train_full) - train_size
train_dataset, val_dataset = random_split(train_full, [train_size, val_size])

# DataLoadery

train_loader = DataLoader(train_dataset,batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# Model

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten() # spłaszczanie do wektoru 1 x 1            
        self.linear_relu_stack = nn.Sequential( 
            nn.Linear(28*28, 512),              
            nn.ReLU(),                          
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = SimpleMLP()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

device = torch.accelerator.current_accelerator().type \
    if torch.accelerator.is_available() else "cpu"

# Funkcje treningowe i walidacyjne

def train_epoch(model, loader):
    # Uzupełnij (ustaw model w tryb treningu)
    total_loss, correct = 0, 0
    model.train()
    for x, y in loader:
        X = x.to(device)
        y = y.to(device)
        logits = model(X)
        pred = logits.argmax(dim=1)
        loss = criterion(logits, y)
        total_loss += loss.item() # bez tego zwracany jest tensor 1 x 1

        correct += (pred == y).sum().item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return total_loss / len(loader), correct / len(loader.dataset)
# 3blue1brown

def eval_epoch(model, loader):
    # Uzupełnij (ustaw model w tryb ewaluacji)
    total_loss, correct = 0, 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            X = x.to(device)
            y = y.to(device)
            logits = model(X)
            pred = logits.argmax(dim=1)
            total_loss += criterion(logits, y).item()
            correct += (pred == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

# Trening z zapisem najlepszego modelu

EPOCHS = 6
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss, val_acc = eval_epoch(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train loss: {train_loss:.4f}, acc: {train_acc:.4f}")
    print(f"  Val loss:   {val_loss:.4f}, acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved PyTorch Model State to best_model.pth")

# Test końcowy: wczytanie najlepszego modelu

model = SimpleMLP().to(device)
model.load_state_dict(torch.load("best_model.pth"))

test_loss, test_acc = eval_epoch(model, test_loader)
print("\nFinal test accuracy:", test_acc)
print(f"Device used: {device}")

Epoch 1/6
  Train loss: 2.1837, acc: 0.3311
  Val loss:   2.0537, acc: 0.5153
Saved PyTorch Model State to best_model.pth
Epoch 2/6
  Train loss: 1.8357, acc: 0.6092
  Val loss:   1.5885, acc: 0.6718
Saved PyTorch Model State to best_model.pth
Epoch 3/6
  Train loss: 1.3219, acc: 0.7332
  Val loss:   1.0978, acc: 0.7672
Saved PyTorch Model State to best_model.pth
Epoch 4/6
  Train loss: 0.9335, acc: 0.7954
  Val loss:   0.8137, acc: 0.8102
Saved PyTorch Model State to best_model.pth
Epoch 5/6
  Train loss: 0.7244, acc: 0.8246
  Val loss:   0.6631, acc: 0.8378
Saved PyTorch Model State to best_model.pth
Epoch 6/6
  Train loss: 0.6093, acc: 0.8452
  Val loss:   0.5749, acc: 0.8512
Saved PyTorch Model State to best_model.pth

Final test accuracy: 0.8566
Device used: cpu


## ⭐️ KMNIST — Kompletny Pipeline od Surowych Danych do Trenowania Sieci

W tym zadaniu pracujesz z datasetem **KMNIST**, ale **bez używania gotowego datasetu z `torchvision`**.

Zrobisz wszystko **ręcznie**:

* pobierzesz dane z `torchvision` (zostało zrobione za Ciebie),
* zapiszesz pojedyncze obrazki jako `.jpg` (zostało zrobione za Ciebie),
* zapiszesz etykiety do plików `.csv` (zostało zrobione za Ciebie),
* zbudujesz własną klasę `Dataset`,
* przygotujesz dataloadery,
* stworzysz i wytrenujesz sieć neuronową opartą o CNN.

---

### Przygotowanie danych

#### Pobranie KMNIST

Zostało zrobione za Ciebie

---

### Twoja własna klasa Dataset

Stwórz klasę dziedziczącą po `torch.utils.data.Dataset`.

#### Wymagania:

#### `__init__`:

* wczytuje odpowiedni plik CSV,
* przechowuje listę ścieżek do obrazów i odpowiadających im etykiet.

#### `__getitem__`:

* otwiera obraz `PIL`,
* stosuje transformacje,
* zwraca:

  ```python
  (tensor, label)
  ```

#### `__len__`:

* zwraca liczbę przykładów.

#### Transformacje

**Obowiązkowe:**

* `transforms.ToTensor()`

**Opcjonalne:**

* `transforms.Normalize((0.5,), (0.5,))`

---

### Podział danych na train / val / test

Dla danych treningowych:

* **90%** → train
* **10%** → validation

Użyj:

```python
torch.utils.data.random_split
```

---

### Dataloadery

Utwórz trzy DataLoadery:

* **train** — `shuffle=True`
* **val** — `shuffle=False`
* **test** — `shuffle=False`

#### Parametry:

* `batch_size = 64`
* `num_workers = 0`

---

### Sieć neuronowa — CNN

Zaimplementuj prosty CNN w PyTorch. Jeśli będziesz miał problem z implementacją napisz maila do prowadzącego - pomożemy.

#### Zalecana architektura:

1. `Conv2d(1, 32, 3, padding=1)`
2. `ReLU()`
3. `MaxPool2d(2)`
4. `Conv2d(32, 64, 3, padding=1)`
5. `ReLU()`
6. `MaxPool2d(2)`
7. `Flatten()`
8. `Linear(64*7*7, 128)`
9. `ReLU()`
10. `Linear(128, 10)`

#### Wskazówki:

* Możesz użyć `nn.Sequential` albo napisać `forward()` ręcznie.

#### Loss i optymalizator:

* **CrossEntropyLoss**
* **Adam**, `lr=0.001`

---

### Pętla treningowa

Dla każdej epoki:

#### W trybie `model.train()`:

Dla każdego batcha:

* oblicz logits (predykcję),
* policz stratę,
* wykonaj:

  * `optimizer.zero_grad()`
  * `loss.backward()`
  * `optimizer.step()`,
* policz accuracy.

#### Po każdej epoce wypisz:

* średni **train loss**
* średnią **val accuracy**

---

### Walidacja i zapis najlepszego modelu

Napisz funkcję ewaluacyjną, która:

* używa `model.eval()`,
* używa `torch.no_grad()`.

Oblicza:

* `val_loss`
* `val_accuracy`

#### Po epce:

Jeśli `val_loss` jest **najlepszy dotychczas**, zapisz model:

```python
torch.save(model.state_dict(), "best_kmnist.pth")
print("Zapisano nowy najlepszy model!")
```

---

### Testowanie

1. Załaduj najlepszy model:

   ```python
   model.load_state_dict(torch.load("best_kmnist.pth"))
   ```

2. Uruchom ewaluację na zbiorze testowym.

3. Wypisz finalne **test accuracy**.


In [1]:
# Komórka odpowiedzialna za przygotowanie danych.
import os
import csv
from torchvision.datasets import KMNIST
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

def export_kmnist(root="kmnist_data"):
    os.makedirs(root, exist_ok=True)

    for split in ["train", "test"]:
        dataset = KMNIST(
            root="./raw_kmnist",
            train=(split=="train"),
            download=True
        )

        img_dir = os.path.join(root, split)
        os.makedirs(img_dir, exist_ok=True)

        csv_path = os.path.join(root, f"{split}_labels.csv")

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["filename", "label"])

            for i, (img, label) in enumerate(dataset):
                filename = f"img_{i:05d}.jpg"
                img_path = os.path.join(img_dir, filename)

                # zapis do jpg
                np_img = np.array(img)
                Image.fromarray(np_img).save(img_path)

                writer.writerow([filename, label])

export_kmnist()


/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: (__ZN3c1017RegisterOperatorsD1Ev)
  Referenced from: '/opt/miniconda3/envs/pytorch_lab/lib/python3.11/site-packages/torchvision/image.so'
  Expected in: '/opt/miniconda3/envs/pytorch_lab/lib/libtorch_cpu.dylib''If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


100.0%


Extracting ./raw_kmnist/KMNIST/raw/train-images-idx3-ubyte.gz to ./raw_kmnist/KMNIST/raw



100.0%


Extracting ./raw_kmnist/KMNIST/raw/train-labels-idx1-ubyte.gz to ./raw_kmnist/KMNIST/raw



100.0%


Extracting ./raw_kmnist/KMNIST/raw/t10k-images-idx3-ubyte.gz to ./raw_kmnist/KMNIST/raw



100.0%


Extracting ./raw_kmnist/KMNIST/raw/t10k-labels-idx1-ubyte.gz to ./raw_kmnist/KMNIST/raw



In [4]:
# Twoje rozwiązanie
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import os

class KMNISTDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform):
        self.labels_df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = self.labels_df.iloc[idx, 0]
        label = self.labels_df.iloc[idx, 1]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('L')

        image = self.transform(image)
        return image, label
    

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

train_dataset = KMNISTDataset(
    csv_file="kmnist_data/train_labels.csv",
    img_dir="kmnist_data/train",
    transform=transform
)

test_dataset = KMNISTDataset(
    csv_file="kmnist_data/test_labels.csv",
    img_dir="kmnist_data/test",
    transform=transform
)

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

batch_size = 64
num_workers = 0

train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

class KMNISTCNN(nn.Module):
    def __init__(self):
        super(KMNISTCNN, self).__init__()
        
        # Warstwy konwolucyjne
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Warstwy w pełni połączone
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    
    return avg_loss, accuracy


device = torch.accelerator.current_accelerator().type \
    if torch.accelerator.is_available() else "cpu"

model = KMNISTCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 4
best_val_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
        
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statystyki
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
    
    avg_train_loss = train_loss / train_total
    train_accuracy = 100.0 * train_correct / train_total
    
    val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}:")
    print(f"  Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}")
    print(f"  Val Accuracy: {val_accuracy:.2f}%")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_kmnist.pth")
        print(f"Zapisano nowy najlepszy model")
    
    print("-" * 30)

print("\nTrening zakończony!")

print("\nTestowanie najlepszego modelu...")
print("-" * 50)

model.load_state_dict(torch.load("best_kmnist.pth"))

test_loss, test_accuracy = evaluate_model(model, test_loader, criterion, device)

print(f"Wyniki na zbiorze testowym:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_accuracy:.2f}%")
print("-" * 50)

Epoch 1:
  Train Loss: 0.2822, Val Loss: 0.1202
  Val Accuracy: 96.05%
Zapisano nowy najlepszy model
------------------------------
Epoch 2:
  Train Loss: 0.0760, Val Loss: 0.0712
  Val Accuracy: 97.82%
Zapisano nowy najlepszy model
------------------------------
Epoch 3:
  Train Loss: 0.0451, Val Loss: 0.0645
  Val Accuracy: 97.88%
Zapisano nowy najlepszy model
------------------------------
Epoch 4:
  Train Loss: 0.0268, Val Loss: 0.0679
  Val Accuracy: 97.97%
------------------------------

Trening zakończony!

Testowanie najlepszego modelu...
--------------------------------------------------
Wyniki na zbiorze testowym:
  Test Loss: 0.2209
  Test Accuracy: 94.31%
--------------------------------------------------
